Nexus: Bot argento convertor de enlaces a video/gif

In [ ]:
# Celda 1: Instalación de librerías y binarios de Playwright
!pip install python-telegram-bot playwright moviepy asyncio nest-asyncio -q
!playwright install chromium
!playwright install-deps

Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2t64 is already the newest version (1.2.11-1ubuntu0.3).
libatk-bridge2.0-0t64 is already the newest version (2.52.0-1build1).
libatk1.

# Celda 2: Imports y parche para el bucle de eventos en Colab

In [ ]:
import os
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes
from moviepy.editor import VideoFileClip

# Aplicar nest_asyncio para permitir bucles asíncronos anidados dentro del entorno de Colab
nest_asyncio.apply()

# Token de autenticación del bot de Telegram
TOKEN = "8823334693:AAH9IEijhk7zEdV6g0ju8HyagCXHaEvY0c8"

# Almacenamiento en memoria de las URLs enviadas por cada usuario (key: user_id, value: url)
user_urls = {}

  from pkg_resources import resource_stream, resource_exists



Celda 3: Funciones auxiliares de captura y conversión

In [ ]:
# Celda 3: Captura vertical con escala y centrado automático

async def capture_webpage_video(url: str, duration_sec: int = 10, output_filename: str = "nexus_story.mp4"):
    """
    Navega a la URL, escala y centra la animación para que luzca perfecta en 9:16.
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            viewport={'width': 1080, 'height': 1920},
            device_scale_factor=1,
            record_video_dir="temp_videos/",
            record_video_size={'width': 1080, 'height': 1920}
        )

        page = await context.new_page()
        await page.goto(url, wait_until="networkidle")

        # Script para centrar y aplicar zoom prolijo sin deformar la animación 3D
        await page.evaluate("""() => {
            document.body.style.margin = '0';
            document.body.style.height = '100vh';
            document.body.style.display = 'flex';
            document.body.style.justifyContent = 'center';
            document.body.style.alignItems = 'center';
            document.body.style.backgroundColor = '#000000';
            document.body.style.overflow = 'hidden';

            const canvas = document.querySelector('canvas');
            if (canvas) {
                // Scalamos el canvas para que se vea bien cerca y con presencia en la Story
                canvas.style.transform = 'scale(1.6)';
                canvas.style.transformOrigin = 'center center';
            }
        }""")

        # Pausa para que la animación renderice fluida
        await asyncio.sleep(2)
        await asyncio.sleep(duration_sec)

        video_path = await page.video.path()
        await context.close()
        await browser.close()

        if os.path.exists(output_filename):
            os.remove(output_filename)
        os.rename(video_path, output_filename)
        return output_filename

def convert_mp4_to_gif(mp4_path: str, gif_path: str = "nexus_story.gif"):
    """
    Convierte el MP4 a GIF optimizado de forma limpia.
    """
    clip = VideoFileClip(mp4_path)
    clip_resized = clip.resize(height=640)
    clip_resized.write_gif(gif_path, fps=15, opt='nq')
    clip.close()
    return gif_path

Celda 4: Handlers (comandos y mensajes del bot)

In [ ]:
# Celda 4: Lógica de respuesta del bot de Telegram

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Responde al comando /start."""
    await update.message.reply_text(
        "¡Bueeenas! 🌸 Soy Nexus ✨\n\n"
        "Pasame el link de tu página o proyecto y te lo grabo al toque. "
        "Contame, ¿qué link vamos a armar hoy?"
    )

async def handle_url(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Valida y guarda la URL provista por el usuario."""
    url = update.message.text.strip()
    user_id = update.effective_user.id

    if not url.startswith("http://") and not url.startswith("https://"):
        await update.message.reply_text(
            "Mmm, che, me parece que a ese link le falta algo 🤔\n"
            "Fijate que empiece con `http://` o `https://` así lo puedo abrir bien."
        )
        return

    user_urls[user_id] = url

    await update.message.reply_text(
        "¡De una, ya agarré tu link! 🌸\n\n"
        "¿En qué formato querés que te lo prepare?\n\n"
        "🎥 Mandame /video (para subirlo directo a tus Stories de IG)\n"
        "✨ O mandame /gif (si querés un GIF livianito para compartir por ahí)"
    )

async def get_video(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Procesa y envía la captura en formato de video MP4."""
    user_id = update.effective_user.id
    url = user_urls.get(user_id)

    if not url:
        await update.message.reply_text("Primero pasame el link de la página que querés que grabe 😉")
        return

    msg = await update.message.reply_text("Dale, bancame unos segundos que abro tu página y te grabo el clip en alta calidad... 🎬")

    try:
        mp4_file = f"video_{user_id}.mp4"
        await capture_webpage_video(url, duration_sec=10, output_filename=mp4_file)

        await msg.edit_text("¡Casi estamos! Ya te lo mando...")

        await update.message.reply_video(
            video=open(mp4_file, 'rb'),
            caption="¡Acá tenés tu video listo para detonar las Stories de Instagram! 💖✨",
            supports_streaming=True
        )
        await msg.delete()

    except Exception as e:
        await msg.edit_text(f"Uhh, saltó un errorcito al renderizar el video 😅: {str(e)}")

async def get_gif(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Procesa, convierte y envía la captura en formato GIF."""
    user_id = update.effective_user.id
    url = user_urls.get(user_id)

    if not url:
        await update.message.reply_text("Primero pasame el link de la página que querés que grabe 😉")
        return

    msg = await update.message.reply_text("¡De una! Bancame un ratito que grabo la animación y te la convierto a GIF... ⚡")

    try:
        mp4_file = f"temp_{user_id}.mp4"
        gif_file = f"gif_{user_id}.gif"

        await capture_webpage_video(url, duration_sec=10, output_filename=mp4_file)
        await msg.edit_text("Procesando el GIF, ya casi sale...")

        convert_mp4_to_gif(mp4_file, gif_file)

        await update.message.reply_animation(
            animation=open(gif_file, 'rb'),
            caption="¡Acá tenés tu GIF bien livianito para mandar a donde quieras! 🌸✨"
        )
        await msg.delete()

    except Exception as e:
        await msg.edit_text(f"Uhh, no pude armar el GIF esta vez 😅: {str(e)}")

Celda 5: Inicialización y ejecución

In [ ]:
# Celda 5: Construcción de la aplicación e inicio de la escucha de mensajes
async def main():
    app = ApplicationBuilder().token(TOKEN).build()

    # Registro de handlers
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("video", get_video))
    app.add_handler(CommandHandler("gif", get_gif))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_url))

    print("🤖 Nexus está activo y escuchando en Telegram...")
    await app.run_polling()

# Ejecución del bucle principal
if __name__ == '__main__':
    await main()

🤖 Nexus está activo y escuchando en Telegram...
MoviePy - Building file gif_6195194238.gif with imageio.


RuntimeError: Cannot close a running event loop